# Скоринг ИП — простой учебный Jupyter Notebook

**Альфа-Хакатон · Аналитик 2 · версия без SQLite и дополнительных библиотек.**

Данные синтетические. Веса, пороги, лимиты, стоп-факторы и правила R1–R12 — **проектные допущения для MVP**, а не настоящая кредитная политика Альфа-Банка. Результат `OFFERED` означает предложение условий, но **не выдачу денег**.

Выполняйте ячейки **сверху вниз**. Для эксперимента достаточно менять словарь `my_ip` в разделе 4.

## 1. Параметры модели

| Фактор | Значения |
|---|---|
| Возраст бизнеса | ≥24 мес.: 20; 6–23: 10; <6: 0 |
| Изменение выручки | >10%: 25; от −10% до +10%: 15; меньше −10%: 0 |
| Долговая нагрузка | ≤30%: 30; от >30% до 50%: 15; >50%: 0 |
| Кредитная история | Чистая: 25; небольшие погашенные просрочки: 10; серьёзные: 0 |

**Важно:** `debt_payments / revenue` — лишь упрощённый показатель нагрузки. Он не отражает расходы, свободный денежный поток и будущий платёж. При `revenue = 0` или предыдущей выручке `0` автоматический расчёт запрещён.

In [1]:
# Шкала и ограничения одного учебного продукта.
MIN_AMOUNT = 100_000
MAX_AMOUNT = 3_000_000
HISTORY_POINTS = {"clean": 25, "minor_closed": 10, "serious": 0}


def risk_group(score):
    """Категория риска по сумме баллов."""
    if not 0 <= score <= 100:
        raise ValueError("Балл должен быть от 0 до 100")
    if score >= 75:
        return "Низкий"
    if score >= 50:
        return "Средний"
    return "Высокий"


def calculate_score(ip):
    """Вернуть балл и баллы по каждому фактору. При неверных данных — ошибка."""
    age = ip["age_months"]
    revenue = ip["revenue"]
    previous = ip["previous_revenue"]
    debt = ip["debt_payments"]
    history = ip["credit_history"]

    if age < 0 or revenue <= 0 or previous <= 0 or debt < 0:
        raise ValueError("Некорректные финансовые данные")
    if history not in HISTORY_POINTS:
        raise ValueError("Неизвестный тип кредитной истории")

    growth = (revenue - previous) / previous * 100
    burden = debt / revenue * 100
    points = {
        "Возраст": 20 if age >= 24 else (10 if age >= 6 else 0),
        "Выручка": 25 if growth > 10 else (15 if growth >= -10 else 0),
        "Долговая нагрузка": 30 if burden <= 30 else (15 if burden <= 50 else 0),
        "Кредитная история": HISTORY_POINTS[history],
    }
    score = sum(points.values())
    return {
        "score": score,
        "risk": risk_group(score),
        "points": points,
        "growth_percent": round(growth, 2),
        "burden_percent": round(burden, 2),
    }

print("Модель загружена: максимум 100 баллов")

Модель загружена: максимум 100 баллов


## 2. Решение — отдельно от скоринга

Порядок правил R1–R11: исправление параметров → стоп-фактор → запрос обязательных данных → техническое ожидание → некорректные вычисления → высокий риск → слишком маленький лимит → автоматическое предложение при низком риске → ручной этап при среднем риске.

**R12** — отдельное решение сотрудника после ручного рассмотрения, поэтому оно не возвращается обратно в R10–R11.

Ограничения: оборотный кредит 100 тыс.–3 млн ₽, срок 6–36 месяцев. Низкий риск: `L = min(3 млн, 3 × месячная выручка)`. Средний риск: `L = min(1 млн, месячная выручка)`.

In [2]:
def evaluate(ip):
    """Однозначное решение по правилам учебного кредитного конвейера."""
    answer = {"score": None, "risk": None, "limit": None,
              "offered_amount": None, "rule": None, "decision": None,
              "points": None, "reason": ""}

    def result(rule, decision, reason, amount=None):
        answer.update(rule=rule, decision=decision, reason=reason,
                      offered_amount=amount)
        return answer

    amount = ip.get("requested_amount")
    term = ip.get("term_months")
    purpose = ip.get("purpose")
    if ((amount is not None and (not isinstance(amount, (int, float)) or not MIN_AMOUNT <= amount <= MAX_AMOUNT))
            or (term is not None and (not isinstance(term, int) or not 6 <= term <= 36))
            or (purpose is not None and purpose != "working_capital")):
        return result("R1", "CORRECTION_REQUIRED", "Исправьте сумму, срок или цель")

    # Стоп-фактор не зависит от суммы баллов.
    if ip.get("active") is False or (ip.get("overdue_days") is not None and ip["overdue_days"] >= 30):
        try:
            diagnostic = calculate_score(ip)
            answer.update(score=diagnostic["score"], risk=diagnostic["risk"],
                          points=diagnostic["points"])
        except (KeyError, TypeError, ValueError, ZeroDivisionError):
            pass
        return result("R2", "REJECTED", "Неактивный ИП или просрочка от 30 дней")

    required = ("age_months", "revenue", "previous_revenue", "debt_payments",
                "credit_history", "active", "overdue_days", "requested_amount",
                "term_months", "purpose", "bki_consent")
    if any(ip.get(field) is None for field in required) or ip.get("bki_consent") is not True:
        return result("R3", "NEED_INFO", "Не хватает данных или согласия на полный отчёт БКИ")
    if ip.get("external_available") is False:
        return result("R4", "WAIT_EXTERNAL", "Заглушка проверки временно недоступна")

    try:
        data = calculate_score(ip)
    except (KeyError, TypeError, ValueError, ZeroDivisionError):
        return result("R5", "MANUAL_REVIEW", "Противоречивые данные или деление на ноль")
    answer.update(score=data["score"], risk=data["risk"], points=data["points"])

    if data["risk"] == "Высокий":
        return result("R6", "REJECTED", "Высокий риск")

    revenue = ip["revenue"]
    limit = min(3_000_000, 3 * revenue) if data["risk"] == "Низкий" else min(1_000_000, revenue)
    answer["limit"] = int(limit)
    if limit < MIN_AMOUNT:
        return result("R7", "REJECTED", "Лимит ниже 100 тыс. ₽")

    # R12: финальное действие кредитного специалиста после ручной проверки.
    manual = ip.get("manual_decision")
    if manual is not None:
        if data["risk"] != "Средний" or manual not in ("approve", "reject"):
            return result("R5", "MANUAL_REVIEW", "Некорректное ручное решение")
        if manual == "reject":
            return result("R12", "REJECTED", "Решение сотрудника: отказ")
        manual_amount = ip.get("manual_amount")
        if (not isinstance(manual_amount, int) or
                not MIN_AMOUNT <= manual_amount <= limit or manual_amount > amount):
            return result("R5", "MANUAL_REVIEW", "Недопустимая сумма после ручной проверки")
        return result("R12", "OFFERED", "Решение сотрудника: предложение", manual_amount)

    if data["risk"] == "Низкий":
        if amount <= limit:
            return result("R8", "OFFERED", "Запрошенная сумма доступна", int(amount))
        return result("R9", "OFFERED", "Предложена меньшая сумма", int(limit))
    if amount <= limit:
        return result("R10", "MANUAL_REVIEW", "Средний риск — нужен сотрудник")
    return result("R11", "MANUAL_REVIEW", "Средний риск, запрос превышает лимит")

print("Decision Table R1–R12 загружена")

Decision Table R1–R12 загружена


## 3. Один предприниматель: меняйте параметры

Это полностью синтетический профиль. `credit_history`: `clean`, `minor_closed`, `serious`. Для ручного одобрения используйте дополнительно `manual_decision = "approve"` и `manual_amount` после завершения проверки сотрудником.

In [3]:
my_ip = {
    "age_months": 36,              # Сколько месяцев существует ИП
    "revenue": 500_000,           # Выручка за месяц
    "previous_revenue": 450_000,  # Среднемесячная выручка за предыдущий период
    "debt_payments": 100_000,     # Платежи по действующим кредитам за месяц
    "credit_history": "clean",   # Нет просрочек
    "active": True,
    "overdue_days": 0,
    "bki_consent": True,
    "external_available": True,
    "requested_amount": 1_000_000,
    "term_months": 12,
    "purpose": "working_capital",
}

answer = evaluate(my_ip)
for key, value in answer.items():
    print(f"{key}: {value}")

score: 100
risk: Низкий
limit: 1500000
offered_amount: 1000000
rule: R8
decision: OFFERED
points: {'Возраст': 20, 'Выручка': 25, 'Долговая нагрузка': 30, 'Кредитная история': 25}
reason: Запрошенная сумма доступна


## 4. Синтетические профили T1–T20

Реальная база не требуется: профили — обычные словари Python. Для краткости новые профили строим на базовом `base_ip`, меняя нужные параметры. Таблица ниже проверяет **рассчитанный результат**, а не просто демонстрирует ожидаемые ответы.

In [4]:
base_ip = {
    "age_months": 48, "revenue": 800_000, "previous_revenue": 680_000,
    "debt_payments": 160_000, "credit_history": "clean",
    "active": True, "overdue_days": 0, "bki_consent": True,
    "external_available": True, "requested_amount": 2_000_000,
    "term_months": 12, "purpose": "working_capital",
}

def make_ip(**changes):
    return {**base_ip, **changes}

tests = [
    ("T1", "Сильный, сумма в лимите", make_ip(), 100, "R8", "OFFERED", 2_000_000),
    ("T2", "Сильный, сумма больше лимита", make_ip(requested_amount=3_000_000), 100, "R9", "OFFERED", 2_400_000),
    ("T3", "Средний, сумма в лимите", make_ip(age_months=18, revenue=600_000, previous_revenue=600_000, debt_payments=240_000, requested_amount=500_000), 65, "R10", "MANUAL_REVIEW", None),
    ("T4", "Средний, запрос выше лимита", make_ip(age_months=18, revenue=600_000, previous_revenue=600_000, debt_payments=240_000, requested_amount=900_000), 65, "R11", "MANUAL_REVIEW", None),
    ("T5", "Высокий риск", make_ip(age_months=4, revenue=300_000, previous_revenue=375_000, debt_payments=180_000, credit_history="minor_closed", requested_amount=200_000), 10, "R6", "REJECTED", None),
    ("T6", "Текущая просрочка 45 дней", make_ip(overdue_days=45, credit_history="serious"), 75, "R2", "REJECTED", None),
    ("T7", "Выручка неизвестна", make_ip(revenue=None), None, "R3", "NEED_INFO", None),
    ("T8", "Ровно 75 баллов", make_ip(revenue=400_000, previous_revenue=400_000, debt_payments=160_000, requested_amount=1_000_000), 75, "R8", "OFFERED", 1_000_000),
    ("T9", "Нижняя достижимая граница 50", make_ip(age_months=12, revenue=300_000, previous_revenue=300_000, debt_payments=120_000, credit_history="minor_closed", requested_amount=250_000), 50, "R10", "MANUAL_REVIEW", None),
    ("T10", "Ближайший балл ниже 50: 45", make_ip(revenue=300_000, previous_revenue=300_000, debt_payments=180_000, credit_history="minor_closed", requested_amount=200_000), 45, "R6", "REJECTED", None),
    ("T11", "Ближайший балл ниже 75: 70", make_ip(debt_payments=320_000, credit_history="minor_closed", requested_amount=500_000), 70, "R10", "MANUAL_REVIEW", None),
    ("T12", "Неактивный ИП", make_ip(active=False), 100, "R2", "REJECTED", None),
    ("T13", "Срок 48 месяцев", make_ip(term_months=48), None, "R1", "CORRECTION_REQUIRED", None),
    ("T14", "Предыдущая выручка нулевая", make_ip(previous_revenue=0), None, "R5", "MANUAL_REVIEW", None),
    ("T15", "Не работает заглушка БКИ", make_ip(external_available=False), None, "R4", "WAIT_EXTERNAL", None),
    ("T16", "Нет согласия на отчёт БКИ", make_ip(bki_consent=False), None, "R3", "NEED_INFO", None),
    ("T17", "Лимит ниже минимума", make_ip(revenue=20_000, previous_revenue=20_000, debt_payments=8_000, requested_amount=100_000), 75, "R7", "REJECTED", None),
    ("T18", "Сумма точно на лимите", make_ip(requested_amount=2_400_000), 100, "R8", "OFFERED", 2_400_000),
    ("T19", "Ручное одобрение сотрудника", make_ip(age_months=18, revenue=600_000, previous_revenue=600_000, debt_payments=240_000, requested_amount=900_000, manual_decision="approve", manual_amount=600_000), 65, "R12", "OFFERED", 600_000),
    ("T20", "Противоречивый отрицательный долг", make_ip(debt_payments=-100), None, "R5", "MANUAL_REVIEW", None),
]

print("Количество тестовых профилей:", len(tests))

Количество тестовых профилей: 20


## 5. Проверка результатов

Зелёная проверка — сработали ожидаемые баллы, правило, решение и сумма. Если бизнес-логику случайно изменили, `assert` покажет ошибку.

In [5]:
print(f"{'ID':<5} {'Балл':<8} {'Правило':<8} {'Решение':<22} {'Сумма':<12} Статус")
print("-" * 75)
for name, description, ip, expected_score, expected_rule, expected_decision, expected_amount in tests:
    actual = evaluate(ip)
    assert actual["score"] == expected_score, f"{name}: балл {actual['score']}"
    assert actual["rule"] == expected_rule, f"{name}: правило {actual['rule']}"
    assert actual["decision"] == expected_decision, f"{name}: решение {actual['decision']}"
    assert actual["offered_amount"] == expected_amount, f"{name}: сумма {actual['offered_amount']}"
    print(f"{name:<5} {str(actual['score']):<8} {actual['rule']:<8} {actual['decision']:<22} {str(actual['offered_amount']):<12} OK")

# У правила R9 дополнительная граница: превышение лимита на один рубль.
assert evaluate(make_ip(requested_amount=2_400_001))["rule"] == "R9"
# Проверяем также чистую функцию классификации на всех четырёх границах.
for score, expected in [(49, "Высокий"), (50, "Средний"), (74, "Средний"), (75, "Низкий")]:
    assert risk_group(score) == expected
print("\nВсе 20 сценариев и дополнительные граничные проверки пройдены")

ID    Балл     Правило  Решение                Сумма        Статус
---------------------------------------------------------------------------
T1    100      R8       OFFERED                2000000      OK
T2    100      R9       OFFERED                2400000      OK
T3    65       R10      MANUAL_REVIEW          None         OK
T4    65       R11      MANUAL_REVIEW          None         OK
T5    10       R6       REJECTED               None         OK
T6    75       R2       REJECTED               None         OK
T7    None     R3       NEED_INFO              None         OK
T8    75       R8       OFFERED                1000000      OK
T9    50       R10      MANUAL_REVIEW          None         OK
T10   45       R6       REJECTED               None         OK
T11   70       R10      MANUAL_REVIEW          None         OK
T12   100      R2       REJECTED               None         OK
T13   None     R1       CORRECTION_REQUIRED    None         OK
T14   None     R5       MANUAL_REVIEW 

## 6. Несколько ИП за один раз — без базы

Все профили можно хранить как список Python. Аналогично позже придут входные параметры из веб-формы и API.

In [6]:
entrepreneurs = [ip for _, _, ip, *_ in tests]
decisions_count = {}
for ip in entrepreneurs:
    decision = evaluate(ip)["decision"]
    decisions_count[decision] = decisions_count.get(decision, 0) + 1
print("Распределение решений по 20 синтетическим профилям:")
for decision, count in sorted(decisions_count.items()):
    print(f"{decision}: {count}")

Распределение решений по 20 синтетическим профилям:
CORRECTION_REQUIRED: 1
MANUAL_REVIEW: 6
NEED_INFO: 2
OFFERED: 5
REJECTED: 5
WAIT_EXTERNAL: 1


## Что дальше?

Позже перенесём функции `risk_group`, `calculate_score`, `evaluate` в один файл `credit_scoring.py`. Бэкенд сможет передавать в `evaluate(ip)` JSON-параметры заявки и получать JSON-результат. Для хакатона **не нужно ни SQLite, ни ML**. Для реального кредитования потребуются юридические проверки, настоящие источники и калибровка модели.